# Manual Track Curation

Workflow:
1. Set `CONFIG_PATH` to the dataset config yaml (see `configs/tracking/`)
2. Load tracks CSV + raw tiffs + instance labels into napari
3. Nuclei labels are **coloured by track ID** (same nucleus = same colour across all timepoints)
4. Use the **Curation GUI dock widget** to add corrections, preview them live, and save a corrected CSV/TIFF

## Correction format

```python
# Swap two track IDs from timepoint t onward
{'type': 'swap', 't_from': 5, 'id_a': 12, 'id_b': 34}

# Reassign a single nucleus at timepoint t from one track to another
{'type': 'reassign', 't': 5, 'from_id': 12, 'to_id': 34}

# Break a track at timepoint t (everything from t onward becomes a new track)
{'type': 'break', 'track_id': 12, 't_break': 10}
```

> **Track IDs in the GUI** are displayed as `track_id + 1` (so background = 0).  
> The GUI subtracts 1 automatically before storing corrections.

In [ ]:
import sys
from pathlib import Path
import yaml

# ── set this to the dataset config you want to work with ──
REPO_ROOT   = Path(__file__).resolve().parents[2] if '__file__' in dir() else Path('/mnt/md1/elysse/code/embryo_image_analysis')
CONFIG_PATH = REPO_ROOT / 'configs' / 'tracking' / 'dataset001_implantation.yaml'

# make src/ importable when running from this notebook
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

RAW_DIR    = Path(cfg['paths']['raw_dir'])
LABEL_DIR  = Path(cfg['paths']['label_dir'])
TRACKS_DIR = Path(cfg['paths']['tracks_dir'])

RAW_GLOB   = cfg['paths']['raw_glob']
LABEL_GLOB = cfg['paths']['label_glob']

N_TIMEPOINTS = cfg['microscopy']['n_timepoints']
VX_Z, VX_Y, VX_X = cfg['microscopy']['voxel_size_zyx']

VERSION     = cfg['tracking']['input_version']
OUT_VERSION = cfg['tracking']['output_version']

print(f'Config:      {CONFIG_PATH}')
print(f'Input CSV:   tracks_{VERSION}.csv')
print(f'Output CSV:  tracks_{OUT_VERSION}.csv')
print(f'Tracks dir:  {TRACKS_DIR}')

In [ ]:
import glob
import numpy as np
import pandas as pd
import napari
from skimage.io import imread, imsave

# Load tracks
tracks_df = pd.read_csv(TRACKS_DIR / f'tracks_{VERSION}.csv')
print(f'Loaded {tracks_df["track_id"].nunique()} tracks, {len(tracks_df)} rows')
print('Columns:', list(tracks_df.columns))

has_orig = 'z_um_orig' in tracks_df.columns
if has_orig:
    print('Original image coordinates found — napari will display in image space.')
else:
    print('WARNING: Original coordinates not found. Displaying registered coords instead.')

z_col, y_col, x_col = ('z_um_orig', 'y_um_orig', 'x_um_orig') if has_orig else ('z_um', 'y_um', 'x_um')

In [ ]:
# Load raw tiffs and instance labels
raw_files   = sorted(glob.glob(str(RAW_DIR   / RAW_GLOB)))[:N_TIMEPOINTS]
label_files = sorted(glob.glob(str(LABEL_DIR / LABEL_GLOB)))[:N_TIMEPOINTS]

assert len(raw_files)   == N_TIMEPOINTS, f'Expected {N_TIMEPOINTS} raw files,   found {len(raw_files)}'
assert len(label_files) == N_TIMEPOINTS, f'Expected {N_TIMEPOINTS} label files, found {len(label_files)}'

print('Loading raw tiffs...',   flush=True)
raw_stack   = np.stack([imread(f) for f in raw_files])
print('Loading label tiffs...', flush=True)
label_stack = np.stack([imread(f) for f in label_files])

print(f'Raw stack:   {raw_stack.shape}')
print(f'Label stack: {label_stack.shape}')

In [ ]:
from src.tracking import build_track_label_stack

print('Building track-coloured label stack...', flush=True)
track_label_stack = build_track_label_stack(label_stack, tracks_df)
print('Done.')

In [ ]:
# Open napari — physical scale (T=1, Z, Y, X in µm)
SCALE = (1, VX_Z, VX_Y, VX_X)

viewer = napari.Viewer()

viewer.add_image(raw_stack, name='raw', scale=SCALE, colormap='gray', opacity=0.8)
viewer.add_labels(track_label_stack, name='track_labels', scale=SCALE, opacity=0.4)
viewer.add_labels(label_stack, name='instance_labels', scale=SCALE, opacity=0.25, blending='translucent')

centroid_coords = tracks_df[['t', z_col, y_col, x_col]].values
viewer.add_points(
    centroid_coords,
    name='centroids',
    features={'track_id': tracks_df['track_id'].values + 1},
    text={'string': '{track_id}', 'size': 8, 'color': 'white', 'anchor': 'center'},
    face_color='track_id',
    face_colormap='husl',
    border_color='black',
    border_width=0.05,
    size=4,
    opacity=0.9,
)

track_data = (
    tracks_df
    .sort_values(['track_id', 't'])
    [['track_id', 't', z_col, y_col, x_col]]
    .values
)
viewer.add_tracks(track_data, name='tracks', tail_length=10, head_length=0, tail_width=3)

print('napari layers: raw | track_labels | instance_labels | centroids | tracks')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Curation GUI — dock widget
# ─────────────────────────────────────────────────────────────
from qtpy.QtWidgets import (
    QWidget, QVBoxLayout, QHBoxLayout, QGroupBox,
    QLabel, QLineEdit, QPushButton, QComboBox,
    QListWidget, QFileDialog, QSizePolicy
)
from qtpy.QtCore import Qt
from src.tracking import apply_corrections, check_duplicates


class CurationWidget(QWidget):
    def __init__(self, viewer, tracks_df, label_stack, build_fn,
                 z_col, y_col, x_col, tracks_dir, out_version):
        super().__init__()
        self.viewer      = viewer
        self.tracks_df   = tracks_df
        self.label_stack = label_stack
        self.build_fn    = build_fn
        self.z_col, self.y_col, self.x_col = z_col, y_col, x_col
        self.tracks_dir  = tracks_dir
        self.out_version = out_version
        self.corrections = []
        self._fields     = {}
        self._build_ui()

    def _build_ui(self):
        root = QVBoxLayout(self)

        type_box = QGroupBox('Add correction')
        type_layout = QVBoxLayout(type_box)
        self.type_combo = QComboBox()
        self.type_combo.addItems(['swap', 'reassign', 'break'])
        self.type_combo.currentTextChanged.connect(self._refresh_fields)
        type_layout.addWidget(self.type_combo)

        self.fields_layout = QVBoxLayout()
        type_layout.addLayout(self.fields_layout)
        self._refresh_fields('swap')

        add_btn = QPushButton('Add correction')
        add_btn.clicked.connect(self._add_correction)
        type_layout.addWidget(add_btn)
        root.addWidget(type_box)

        list_box = QGroupBox('Pending corrections')
        lbl = QVBoxLayout(list_box)
        self.corr_list = QListWidget()
        self.corr_list.setMaximumHeight(160)
        lbl.addWidget(self.corr_list)
        rm_btn = QPushButton('Remove selected')
        rm_btn.clicked.connect(self._remove_correction)
        lbl.addWidget(rm_btn)
        root.addWidget(list_box)

        self.preview_btn = QPushButton('Preview in napari')
        self.preview_btn.clicked.connect(self._preview)
        root.addWidget(self.preview_btn)

        self.save_csv_btn = QPushButton('Save corrected CSV')
        self.save_csv_btn.clicked.connect(self._save_csv)
        root.addWidget(self.save_csv_btn)

        self.save_tif_btn = QPushButton('Save corrected label TIFF')
        self.save_tif_btn.clicked.connect(self._save_tiff)
        root.addWidget(self.save_tif_btn)

        self.status = QLabel('')
        self.status.setWordWrap(True)
        root.addWidget(self.status)
        root.addStretch()

    FIELD_SPECS = {
        'swap':     [('t_from', 'Start timepoint'), ('id_a', 'Track ID A (as shown)'), ('id_b', 'Track ID B (as shown)')],
        'reassign': [('t',      'Start timepoint'), ('from_id', 'From ID (as shown)'), ('to_id', 'To ID (as shown)')],
        'break':    [('track_id', 'Track ID (as shown)'), ('t_break', 'Break at timepoint')],
    }
    _TRACK_ID_KEYS = {'id_a', 'id_b', 'from_id', 'to_id', 'track_id'}

    def _refresh_fields(self, ctype):
        while self.fields_layout.count():
            item = self.fields_layout.takeAt(0)
            if item.widget():
                item.widget().deleteLater()
        self._fields = {}
        for key, label in self.FIELD_SPECS.get(ctype, []):
            row = QHBoxLayout()
            row.addWidget(QLabel(f'{label}:'))
            edit = QLineEdit()
            edit.setPlaceholderText('int')
            row.addWidget(edit)
            self._fields[key] = edit
            container = QWidget()
            container.setLayout(row)
            self.fields_layout.addWidget(container)

    def _add_correction(self):
        ctype = self.type_combo.currentText()
        try:
            corr = {'type': ctype}
            for key, edit in self._fields.items():
                val = int(edit.text())
                if key in self._TRACK_ID_KEYS:
                    val -= 1  # displayed IDs are track_id+1; CSV stores raw track_id
                corr[key] = val
        except ValueError:
            self.status.setText('All fields must be integers.')
            return
        self.corrections.append(corr)
        self.corr_list.addItem(str(corr))
        for e in self._fields.values():
            e.clear()
        self.status.setText(f'{len(self.corrections)} correction(s) pending.')

    def _remove_correction(self):
        row = self.corr_list.currentRow()
        if row >= 0:
            self.corr_list.takeItem(row)
            self.corrections.pop(row)
        self.status.setText(f'{len(self.corrections)} correction(s) pending.')

    def _get_corrected_df(self):
        return apply_corrections(self.tracks_df, self.corrections)

    def _preview(self):
        self.status.setText('Building preview...')
        try:
            cdf = self._get_corrected_df()
            dups = check_duplicates(cdf)
            if not dups.empty:
                disp = dups.copy()
                disp['track_id'] = disp['track_id'] + 1
                disp = disp.rename(columns={'track_id': 'track_id (displayed)'})
                msg = f'{len(dups)} duplicate (track_id, t) pair(s) found — fix before saving:\n'
                msg += disp.head(10).to_string(index=False)
                self.status.setText(msg)
                return

            self.viewer.layers['track_labels'].data = self.build_fn(self.label_stack, cdf)
            self.viewer.layers['track_labels'].refresh()

            pts = self.viewer.layers['centroids']
            pts.data = cdf[['t', self.z_col, self.y_col, self.x_col]].values
            pts.features = {'track_id': cdf['track_id'].values + 1}
            pts.face_color = 'track_id'

            n_before = self.tracks_df['track_id'].nunique()
            n_after  = cdf['track_id'].nunique()
            self.status.setText(f'Preview updated. Tracks: {n_before} -> {n_after}')
        except Exception as e:
            self.status.setText(f'Error: {e}')

    def _save_csv(self):
        default = str(self.tracks_dir / f'tracks_{self.out_version}.csv')
        path, _ = QFileDialog.getSaveFileName(self, 'Save corrected CSV', default, 'CSV files (*.csv)')
        if not path:
            return
        try:
            cdf = self._get_corrected_df()
            dups = check_duplicates(cdf)
            if not dups.empty:
                self.status.setText(f'Save blocked: {len(dups)} duplicate (track_id, t) pair(s). Run preview to see details.')
                return
            cdf.to_csv(path, index=False)
            self.status.setText(f'CSV saved: {path}')
        except Exception as e:
            self.status.setText(f'Error: {e}')

    def _save_tiff(self):
        default = str(self.tracks_dir / f'track_labels_{self.out_version}.tif')
        path, _ = QFileDialog.getSaveFileName(self, 'Save corrected label TIFF', default, 'TIFF files (*.tif *.tiff)')
        if not path:
            return
        self.status.setText('Building corrected label stack...')
        try:
            cdf = self._get_corrected_df()
            dups = check_duplicates(cdf)
            if not dups.empty:
                self.status.setText(f'Save blocked: {len(dups)} duplicate (track_id, t) pair(s). Run preview to see details.')
                return
            imsave(path, self.build_fn(self.label_stack, cdf))
            self.status.setText(f'TIFF saved: {path}')
        except Exception as e:
            self.status.setText(f'Error: {e}')


# ── attach widget to napari ──
curation_widget = CurationWidget(
    viewer      = viewer,
    tracks_df   = tracks_df,
    label_stack = label_stack,
    build_fn    = build_track_label_stack,
    z_col=z_col, y_col=y_col, x_col=x_col,
    tracks_dir  = TRACKS_DIR,
    out_version = OUT_VERSION,
)
viewer.window.add_dock_widget(curation_widget, name='Track Curation', area='right')
print('Curation widget docked.')

## Quick diagnostics

Run after saving the corrected CSV to verify track quality.

In [ ]:
import matplotlib.pyplot as plt

corrected_df = pd.read_csv(TRACKS_DIR / f'tracks_{OUT_VERSION}.csv')
lengths = corrected_df.groupby('track_id')['t'].count().sort_values(ascending=False)

t_start = cfg['tracking']['t_start']
t_end   = cfg['tracking']['t_end']
curated_len = t_end - t_start + 1

print(f'Full timecourse tracks ({N_TIMEPOINTS} frames): {(lengths == N_TIMEPOINTS).sum()}')
print(f'Covers curated range  ({curated_len} frames, t{t_start}-t{t_end}): {(lengths == curated_len).sum()}')
print(f'>= 50 frames: {(lengths >= 50).sum()}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=30)
ax.set_xlabel('Track length (frames)')
ax.set_ylabel('Count')
ax.set_title(f'Track length distribution — {OUT_VERSION}')
ax.axvline(50, color='red', linestyle='--', label='50 frames')
ax.legend()
plt.show()